# 01 — Build trustworthy reference data

> Why did Marketplace revenue decline last month?

Lesson 00 starts with supplied reference facts, authored candidate answers, and authored narrative reviews. Here we derive the numerical reference from synthetic postings and check it against two calculations and a manual reconciliation. That gives the evaluator reproducible evidence for the size of the decline.

Open this notebook with the repository's virtual environment and run it from the top. See the [companion guide](../docs/lessons/01_reference_data.md) and [environment setup](../docs/setup.md).

In [1]:
import inspect
import json
import sqlite3
import subprocess
import sys
from contextlib import closing
from datetime import date
from decimal import Decimal
from pathlib import Path

from IPython.display import Markdown, display
from finance_evals.contracts import evaluate
from finance_evals.reference_calculations import (
    PeriodRevenue, aggregate_revenue_sql, compare_revenue,
    completed_months, reconcile_revenue_python,
)
from finance_evals.source_data import load_revenue_transactions

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
FIXTURE = ROOT / "evals/source/revenue_transactions_v1.json"
print("Kernel:", sys.executable)
print("Project:", ROOT)

Kernel: /home/vammun01/projects/eval-ai/.venv/bin/python
Project: /home/vammun01/projects/eval-ai


## 1. Inspect the source schema and all eight postings

The loader builds SQLite's `revenue_transactions` table. `transaction_id` is the unique key, `posting_date` assigns the calendar month, and `business_segment` scopes the report. The currency is USD throughout. `amount_cents` stores an integer magnitude. For example, 100 cents is $1.00. `entry_type` determines whether to add a revenue or subtract a credit.

In [2]:
def show_table(headers, rows):
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
        *("| " + " | ".join(map(str, row)) + " |" for row in rows),
    ]
    display(Markdown("\n".join(lines)))


def usd(cents):
    return f"${Decimal(cents) / 100:,.2f}"


with closing(sqlite3.connect(":memory:")) as db:
    load_revenue_transactions(db, FIXTURE)
    schema = db.execute("PRAGMA table_info(revenue_transactions)").fetchall()
    postings = db.execute("SELECT * FROM revenue_transactions ORDER BY transaction_id").fetchall()

assert len(postings) == 8
show_table(("Column", "SQLite type", "Required", "Primary key"),
           [(name, storage, bool(required), bool(key))
            for _, name, storage, required, _, key in schema])
show_table((*[column[1] for column in schema], "USD magnitude"),
           [(*row, usd(row[4])) for row in postings])

| Column | SQLite type | Required | Primary key |
| --- | --- | --- | --- |
| transaction_id | TEXT | True | True |
| posting_date | TEXT | True | False |
| business_segment | TEXT | True | False |
| entry_type | TEXT | True | False |
| amount_cents | INTEGER | True | False |
| currency | TEXT | True | False |

| transaction_id | posting_date | business_segment | entry_type | amount_cents | currency | USD magnitude |
| --- | --- | --- | --- | --- | --- | --- |
| TX-001 | 2026-09-01 | Marketplace | revenue | 1100000000 | USD | $11,000,000.00 |
| TX-002 | 2026-09-30 | Marketplace | credit | 100000000 | USD | $1,000,000.00 |
| TX-003 | 2026-10-01 | Marketplace | revenue | 950000000 | USD | $9,500,000.00 |
| TX-004 | 2026-10-31 | Marketplace | credit | 50000000 | USD | $500,000.00 |
| TX-005 | 2026-11-01 | Marketplace | revenue | 200000000 | USD | $2,000,000.00 |
| TX-006 | 2026-09-15 | Direct | revenue | 25000000 | USD | $250,000.00 |
| TX-007 | 2026-10-12 | Direct | revenue | 15000000 | USD | $150,000.00 |
| TX-008 | 2026-10-25 | Direct | credit | 15000000 | USD | $150,000.00 |

## 2. Choose the two completed months and predict the evidence

On November 18, 2026, October is the most recent completed month and September is its comparison month. Include each start date and exclude each end date. This assigns September 30 to September, October 1 and October 31 to October, and leaves November 1 for a later report.

Expect September to use `TX-001` and `TX-002`, and October to use `TX-003` and `TX-004`. Exclude `TX-005` because November is incomplete. Exclude `TX-006`, `TX-007`, and `TX-008` because Direct is outside Marketplace scope. Even though the two October Direct rows cancel out, including them would give the wrong count and evidence IDs.

In [3]:
fixed_today = date.fromisoformat("2026-11-18")
periods = completed_months(fixed_today)
assert [(p.start.isoformat(), p.end.isoformat()) for p in periods] == [
    ("2026-09-01", "2026-10-01"), ("2026-10-01", "2026-11-01"),
]
show_table(("Period", "Inclusive start", "Exclusive end"),
           [(p.label, p.start, p.end) for p in periods])

| Period | Inclusive start | Exclusive end |
| --- | --- | --- |
| 2026-09 | 2026-09-01 | 2026-10-01 |
| 2026-10 | 2026-10-01 | 2026-11-01 |

## 3. Execute the SQL aggregation

`aggregate_revenue_sql` binds the segment, USD currency, and date bounds as parameters. It returns the record count, separate revenue and credit subtotals, net cents, and the sorted IDs behind the results. The in-memory database closes after the queries.

In [4]:
with closing(sqlite3.connect(":memory:")) as db:
    load_revenue_transactions(db, FIXTURE)
    sql = [aggregate_revenue_sql(db, "Marketplace", p) for p in periods]
    direct = [aggregate_revenue_sql(db, "Direct", p) for p in periods]
    wholesale = aggregate_revenue_sql(db, "Wholesale", periods[1])


def money(cents):
    if cents is None:
        return "None"
    return f"{cents:,} ({usd(cents)})"


def show_totals(results):
    show_table(("Period", "Count", "Revenue cents (USD)", "Credit cents (USD)",
                "Net cents (USD)", "Transaction IDs"),
               [(period.label, result.record_count, money(result.revenue_cents),
                 money(result.credit_cents), money(result.net_cents),
                 ", ".join(result.transaction_ids))
                for period, result in zip(periods, results)])


show_totals(sql)

| Period | Count | Revenue cents (USD) | Credit cents (USD) | Net cents (USD) | Transaction IDs |
| --- | --- | --- | --- | --- | --- |
| 2026-09 | 2 | 1,100,000,000 ($11,000,000.00) | 100,000,000 ($1,000,000.00) | 1,000,000,000 ($10,000,000.00) | TX-001, TX-002 |
| 2026-10 | 2 | 950,000,000 ($9,500,000.00) | 50,000,000 ($500,000.00) | 900,000,000 ($9,000,000.00) | TX-003, TX-004 |

## 4. Reconcile independently in Python and check the manual values

Python reads the original JSON with the same validation as the loader, then applies the filters and totals in Python without querying SQLite. Both paths implement the same business specification and use the same period boundaries. If the reporting policy itself is wrong, the two calculations can agree. Compare against the manually selected postings and independently check the period dates.

In [5]:
python = [reconcile_revenue_python(FIXTURE, "Marketplace", p) for p in periods]
manual = [
    PeriodRevenue(2, 1_100_000_000, 100_000_000, 1_000_000_000, ("TX-001", "TX-002")),
    PeriodRevenue(2, 950_000_000, 50_000_000, 900_000_000, ("TX-003", "TX-004")),
]
assert sql == python == manual
show_totals(python)

change = compare_revenue(sql[0].net_cents, sql[1].net_cents)
assert (change.change_cents, change.change_pct, change.reason) == (-100_000_000, "-10.00", None)
print("Signed change:", money(change.change_cents), f"({change.change_pct}%)")

| Period | Count | Revenue cents (USD) | Credit cents (USD) | Net cents (USD) | Transaction IDs |
| --- | --- | --- | --- | --- | --- |
| 2026-09 | 2 | 1,100,000,000 ($11,000,000.00) | 100,000,000 ($1,000,000.00) | 1,000,000,000 ($10,000,000.00) | TX-001, TX-002 |
| 2026-10 | 2 | 950,000,000 ($9,500,000.00) | 50,000,000 ($500,000.00) | 900,000,000 ($9,000,000.00) | TX-003, TX-004 |

Signed change: -100,000,000 ($-1,000,000.00) (-10.00%)


## 5. Distinguish zero from missing data

| October query | Matching postings | Revenue | Credits | Net revenue | Evidence IDs |
| --- | ---: | ---: | ---: | ---: | --- |
| Direct | 2 | $150,000 | $150,000 | $0 | TX-007, TX-008 |
| Wholesale | 0 | `None` | `None` | `None` | `()` |

`record_count` counts postings that match the filters. No posting in the
fixture uses the Wholesale label, so that query matches zero rows. A
segment with postings in other months and none in October would give
the same October result. For Direct, the two amounts offset each other,
so the matching records establish a genuine zero.

### Compare two months

The signed change is current net revenue minus prior net revenue. When
the prior net is positive, percentage change divides that difference by
the prior net and multiplies by 100. The code stores the percentage as a
string rounded to two decimal places, with ties rounded away from zero.

- **Zero prior net.** Going from $0 to $50,000 gives a $50,000 increase.
  Dividing by zero is undefined, so the percentage is `None`.
- **Negative prior net.** Going from −$20,000 to $50,000 gives a $70,000
  increase. Dividing by −$20,000 and multiplying by 100 gives −350%,
  which reads like a decline. The lesson's reporting rule returns
  `None` for the percentage and keeps the $70,000 increase.
- **Missing either month.** The calculation needs both net amounts to
  subtract, so both change fields are `None`.

Run the next cell to compare Direct's two months and three hypothetical inputs. The displayed dollar amounts come from integer cents.

In [6]:
assert direct[1] == reconcile_revenue_python(FIXTURE, "Direct", periods[1])
assert direct[1] == PeriodRevenue(2, 15_000_000, 15_000_000, 0, ("TX-007", "TX-008"))
assert wholesale == reconcile_revenue_python(FIXTURE, "Wholesale", periods[1])
assert wholesale == PeriodRevenue(0, None, None, None, ())

examples = [
    ("Direct, September to October", direct[0].net_cents, direct[1].net_cents, -25_000_000, "-100.00"),
    ("Zero prior net", 0, 5_000_000, 5_000_000, None),
    ("Negative prior net", -2_000_000, 5_000_000, 7_000_000, None),
    ("Missing current month", 25_000_000, None, None, None),
]
rows = []
for scenario, prior, current, expected_change, expected_pct in examples:
    result = compare_revenue(prior, current)
    assert (result.change_cents, result.change_pct) == (expected_change, expected_pct)
    rows.append((
        scenario, money(prior), money(current), money(result.change_cents),
        repr(result.change_pct), result.reason or "—",
    ))
show_table(("Scenario", "Prior net", "Current net", "Signed change", "Stored percentage", "Reason"), rows)

Direct October: PeriodRevenue(record_count=2, revenue_cents=15000000, credit_cents=15000000, net_cents=0, transaction_ids=('TX-007', 'TX-008'))
Wholesale October: PeriodRevenue(record_count=0, revenue_cents=None, credit_cents=None, net_cents=None, transaction_ids=())
Direct comparison: RevenueComparison(change_cents=-25000000, change_pct='-100.00', reason=None)
Missing period: RevenueComparison(change_cents=None, change_pct=None, reason='Missing revenue in one or both periods.')
Zero prior: RevenueComparison(change_cents=200, change_pct=None, reason='Prior net revenue is zero.')
Negative prior: RevenueComparison(change_cents=250, change_pct=None, reason='Prior net revenue is negative.')


## 6. Validate the versioned reference artifact

The nested `reference` holds the exact fields Lesson 00 compares: scope, metric, periods, integer cents, and a formatted percentage string. The outer fields identify the case and its version, fixed date, dataset and business-rule versions, source fixture and IDs, and the evidence limitation. `demo_authored` identifies the demo's review status. The artifact belongs to the evaluator.

The existing generator runs without its `--write` flag. It verifies the SQL, Python, and manual results and prints JSON for comparison with the stored artifact.

In [7]:
artifact_path = ROOT / "evals/reference/lesson01_marketplace_revenue_v1.json"
stored_json = artifact_path.read_text(encoding="utf-8")
verified_json = subprocess.run(
    [sys.executable, str(ROOT / "scripts/generate_lesson01_reference.py")],
    check=True, capture_output=True, text=True, encoding="utf-8",
).stdout
assert stored_json == verified_json, "The stored reference differs from verified generation."
artifact = json.loads(stored_json)
assert artifact["source_record_ids"] == {
    period.label: list(result.transaction_ids) for period, result in zip(periods, sql)
}
assert artifact["review_status"] == "demo_authored"
print(stored_json)

{
  "case_id": "lesson01_marketplace_revenue",
  "version": "1",
  "dataset_version": "1",
  "source_fixture": "evals/source/revenue_transactions_v1.json",
  "business_rule_version": "1",
  "fixed_today": "2026-11-18",
  "review_status": "demo_authored",
  "reference": {
    "scope": "Marketplace",
    "metric": "net_revenue",
    "prior_period": "2026-09",
    "current_period": "2026-10",
    "prior_cents": 1000000000,
    "current_cents": 900000000,
    "change_cents": -100000000,
    "change_pct": "-10.00"
  },
  "source_record_ids": {
    "2026-09": [
      "TX-001",
      "TX-002"
    ],
    "2026-10": [
      "TX-003",
      "TX-004"
    ]
  },
  "evidence_limit": "Posting records establish the amount of the decline. They do not establish its business cause."
}



## 7. Reuse Lesson 00's evaluator and authored reviews

Read the evaluator and inspect each candidate and its review before running the checks. The review must match both the answer ID and the exact narrative. Predict what will happen when both answers match the reference fields but only one discloses the evidence limitation.

In [8]:
print(inspect.getsource(evaluate))
candidates = json.loads((ROOT / "evals/cases/lesson00_answers.json").read_text(encoding="utf-8"))
reviews = json.loads((ROOT / "evals/reviews/lesson00_reviews.json").read_text(encoding="utf-8"))
candidate_by_id = {candidate["id"]: candidate for candidate in candidates}
review_by_id = {review["answer_id"]: review for review in reviews}
selected_ids = ("limited_evidence", "unsupported_churn")
for answer_id in selected_ids:
    candidate, review = candidate_by_id[answer_id], review_by_id[answer_id]
    assert candidate["narrative"] == review["narrative"]
    assert review["review_status"] == "demo_authored"
    print(answer_id, candidate["narrative"], sep="\n")
    print(f"Authored review: {review['verdict']} — {review['reason']}\n")

def evaluate(answer, reference, narrative_review=None):
    fields = ("prior_cents", "current_cents", "change_cents", "change_pct")
    numbers_ok = all(type(answer.get(k)) is type(reference[k]) and answer[k] == reference[k] for k in fields)
    scope_ok = all(answer.get(k) == reference[k] for k in ("scope", "metric", "prior_period", "current_period"))
    review_ok = (
        isinstance(narrative_review, dict)
        and narrative_review.get("answer_id") == answer.get("id")
        and narrative_review.get("narrative") == answer.get("narrative")
        and narrative_review.get("verdict") in ("pass", "fail")
        and bool(narrative_review.get("reason"))
    )
    grounding = narrative_review["verdict"] if review_ok else "unverifiable"
    checks = {
        "numbers_correct": "pass" if numbers_ok else "fail",
        "scope_and_period_correct": "pass" if scope_ok else "fail",
        "claims_supported": grounding,
    }
    overall = "fail" if "fail" in checks.values() else ("pas

In [9]:
reports = [
    evaluate(candidate_by_id[answer_id], artifact["reference"], review_by_id[answer_id])
    for answer_id in selected_ids
]
assert all(report["checks"]["numbers_correct"] == "pass" for report in reports)
assert all(report["checks"]["scope_and_period_correct"] == "pass" for report in reports)
assert [report["checks"]["claims_supported"] for report in reports] == ["pass", "fail"]
assert [report["overall"] for report in reports] == ["pass", "fail"]
print(json.dumps(reports, indent=2))

[
  {
    "answer_id": "limited_evidence",
    "checks": {
      "numbers_correct": "pass",
      "scope_and_period_correct": "pass",
      "claims_supported": "pass"
    },
    "overall": "pass",
    "review_reason": "The narrative describes the supplied totals and discloses the evidence limit.",
    "mode": "offline_authored_examples"
  },
  {
    "answer_id": "unsupported_churn",
    "checks": {
      "numbers_correct": "pass",
      "scope_and_period_correct": "pass",
      "claims_supported": "fail"
    },
    "overall": "fail",
    "review_reason": "Monthly totals do not establish churn as the cause.",
    "mode": "offline_authored_examples"
  }
]


## 8. Interpret what passed

Both candidates match a numerical reference now supported by specific postings. The narrative verdicts come from supplied authored reviews. This is an offline demonstration; a live agent and an LLM judge are outside its scope. Postings show how revenue and credits contributed to the net decline. A churn explanation would require customer activity or cancellation evidence linked to the decline.

1. Why does including the November 1 posting or the October Direct postings invalidate the evidence, even when an extra pair nets to zero?
2. How do the result fields distinguish Direct's zero from Wholesale's missing data, and what happens to each comparison?
3. Why do both candidates pass the numbers check while only one passes overall? What additional evidence would help investigate churn?

**Next:** Lesson 02 plans to run a finance agent against source data and capture its tool results and trace for evaluation.